# 05access: walking minutes to everyday services (Stage 2, part 3 of 4)

For every hexagon: how many minutes' walk to the nearest clinic, school, park, grocery and rail station,
along real streets and footpaths. Then the **15-minute score**: how many of the five you can reach in 15 minutes.

How it works:
1. The walking network becomes a big table of "who is connected to whom, and how far" (a sparse matrix).
2. Every hexagon centre and every service is joined to its nearest point on the network.
3. For each service type, one shortest-path search (Dijkstra) starts from **all** services of that type at once,
   so every point on the network learns its distance to the nearest one. Five searches in total.
4. The straight-line distance is calculated too, to show where the walk is much longer than it looks.

- Input: `hex_grid.parquet`, `services.parquet`, `walk_nodes.parquet`, `walk_edges.parquet`, `walk_meta.json`
- Output: `data/processed/hex_access.parquet`, `data/processed/access_summary.json`
- Next: `06buildaccess.ipynb`

## Section 0: settings

In [1]:
from pathlib import Path

BASE = Path.cwd()
PROCESSED = BASE / "data" / "processed"

CATEGORIES = ["healthcare", "school", "park", "grocery", "rail"]
WALK_SPEED_KMH = 4.8       # an ordinary walking pace
THRESHOLD_MIN = 15         # "within reach" means a walk of 15 minutes or less (about 1.2 km)
SEARCH_LIMIT_M = 6000      # stop searching beyond 6 km (75 minutes); farther counts as "over 75 min"
METRIC_CRS = 32647

In [2]:
import json, time
import numpy as np
import pandas as pd
from pyproj import Transformer
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import dijkstra
from scipy.spatial import cKDTree

grid = pd.read_parquet(PROCESSED / "hex_grid.parquet")
services = pd.read_parquet(PROCESSED / "services.parquet")
nodes = pd.read_parquet(PROCESSED / "walk_nodes.parquet")
edges = pd.read_parquet(PROCESSED / "walk_edges.parquet")
meta = json.loads((PROCESSED / "walk_meta.json").read_text())

M_PER_MIN = WALK_SPEED_KMH * 1000 / 60
print(f"{len(grid):,} hexagons | {len(services):,} service points | network {len(nodes):,} points, "
      f"{len(edges):,} pieces (OSM as of {meta['osm_date'][:10]})")
print(f"{THRESHOLD_MIN} minutes at {WALK_SPEED_KMH} km/h = {THRESHOLD_MIN * M_PER_MIN:,.0f} m")

4,003 hexagons | 13,169 service points | network 1,263,864 points, 1,400,833 pieces (OSM as of 2026-10-06)
15 minutes at 4.8 km/h = 1,200 m


## Step 1: the network as a sparse matrix

Row = from point, column = to point, value = metres. Each street piece is added in both directions.

In [3]:
n = len(nodes)
u, v, w = edges["u"].values, edges["v"].values, edges["length_m"].values.astype(np.float64)
rows, cols, data = np.r_[u, v], np.r_[v, u], np.r_[w, w]
network = coo_matrix((data, (rows, cols)), shape=(n, n)).tocsr()
print("Matrix:", network.shape, f"| {network.nnz:,} connections")

Matrix: (1263864, 1263864) | 2,801,098 connections


## Step 2: join hexagon centres and services to the nearest network point

Distances are measured in metres (UTM), using a k-d tree for speed.
The short walk from the hexagon centre (or the service) to the street is added to the route.

In [4]:
to_m = Transformer.from_crs(4326, METRIC_CRS, always_xy=True)
nx, ny = to_m.transform(nodes["lon"].values, nodes["lat"].values)
tree = cKDTree(np.c_[nx, ny])

hx, hy = to_m.transform(grid["lon"].values, grid["lat"].values)
hex_gap, hex_node = tree.query(np.c_[hx, hy])
sx, sy = to_m.transform(services["lon"].values, services["lat"].values)
svc_gap, svc_node = tree.query(np.c_[sx, sy])

print("Hexagon centre to nearest street: median %.0f m, 95%% within %.0f m" % tuple(np.percentile(hex_gap, [50, 95])))
print("Service to nearest street:        median %.0f m, 95%% within %.0f m" % tuple(np.percentile(svc_gap, [50, 95])))

Hexagon centre to nearest street: median 69 m, 95% within 1795 m
Service to nearest street:        median 19 m, 95% within 86 m


## Step 3: one shortest-path search per service type

Trick: an extra "virtual" point is connected to every service of one type (with the service-to-street gap as length).
One Dijkstra search from that virtual point gives every network point its distance to the **nearest** service.

In [5]:
result = grid[["h3", "district", "lat", "lon", "population"]].copy()
coo = network.tocoo()
start = time.time()
for cat in CATEGORIES:
    m = (services["category"] == cat).values
    src = n                                     # the virtual point gets number n
    A = coo_matrix((np.r_[coo.data, svc_gap[m] + 0.01], (np.r_[coo.row, np.full(m.sum(), src)], np.r_[coo.col, svc_node[m]])), shape=(n + 1, n + 1)).tocsr()
    dist = dijkstra(A, directed=True, indices=src, limit=SEARCH_LIMIT_M)
    walk_m = hex_gap + dist[hex_node]                       # inf when farther than SEARCH_LIMIT_M
    line_m, _ = cKDTree(np.c_[sx[m], sy[m]]).query(np.c_[hx, hy])
    result[f"{cat}_min"] = np.round(walk_m / M_PER_MIN, 1)
    result[f"{cat}_line_m"] = np.round(line_m)
    print(f"{cat:11s} {m.sum():6,d} points | done in {time.time() - start:5.1f} s")

result = result.replace([np.inf], np.nan)          # NaN = more than SEARCH_LIMIT_M away

healthcare   1,320 points | done in   0.3 s
school       1,476 points | done in   0.7 s
park         8,432 points | done in   1.2 s
grocery      1,742 points | done in   1.5 s
rail           199 points | done in   1.8 s


## Step 4: the 15-minute score

In [6]:
within = pd.DataFrame({c: result[f"{c}_min"] <= THRESHOLD_MIN for c in CATEGORIES})
result["score"] = within.sum(axis=1).astype(int)
result["reach"] = within.apply(lambda r: ", ".join(c for c in CATEGORIES if r[c]) or "none", axis=1)

people = result["population"].sum()
by_score = result.groupby("score")["population"].sum() / people * 100
print("Share of people by score (0 = none of the five within 15 minutes, 5 = all five):")
display(by_score.round(1).rename("% of people").to_frame().T)

Share of people by score (0 = none of the five within 15 minutes, 5 = all five):


score,0,1,2,3,4,5
% of people,15.5,14.6,15.7,19.0,25.1,10.1


In [7]:
# Share of people within 15 minutes of each service, overall and by district
share = {c: (result.loc[within[c], "population"].sum() / people * 100) for c in CATEGORIES}
print("Share of people within 15 minutes' walk:")
display(pd.Series(share).round(1).rename("%").to_frame().T)

district_table = (result.assign(**{c: within[c] * result["population"] for c in CATEGORIES}).groupby("district")[CATEGORIES + ["population"]].sum())
district_table = (district_table[CATEGORIES].div(district_table["population"], axis=0) * 100).round(0)
district_table["average score"] = (result.groupby("district").apply(lambda d: np.average(d["score"], weights=d["population"])
                                                if d["population"].sum() > 0 else np.nan)).round(2)
print("By district (% of people within 15 minutes):")
display(district_table.sort_values("average score", ascending=False))

Share of people within 15 minutes' walk:


,healthcare,school,park,grocery,rail
%,46.8,57.1,65.6,66.5,17.9


By district (% of people within 15 minutes):


,healthcare,school,park,grocery,rail,average score
district,,,,,,
W.P. Kuala Lumpur,76.0,80.0,77.0,86.0,34.0,3.53
Petaling,44.0,53.0,73.0,66.0,21.0,2.57
Ulu Langat,43.0,58.0,68.0,69.0,12.0,2.50
Gombak,33.0,48.0,54.0,61.0,5.0,2.01
Klang,28.0,41.0,41.0,49.0,7.0,1.67
Sepang,19.0,37.0,44.0,34.0,1.0,1.35
W.P. Putrajaya,14.0,27.0,69.0,15.0,2.0,1.26


## Step 5: walk versus straight line

How much longer is the walk than the straight line? A typical urban walk is longer because streets are not
straight and blocks must be walked around. This is only a summary here: the per-hexagon values are a rough
measure (the hexagon centre is often away from a street), so we do not map "barriers" from them.

In [8]:
detour = {}
for c in CATEGORIES:
    walk_m = result[f"{c}_min"] * M_PER_MIN
    line_m = result[f"{c}_line_m"].clip(lower=150)
    near = result[f"{c}_line_m"] <= THRESHOLD_MIN * M_PER_MIN
    detour[c] = round(float(np.nanmedian((walk_m / line_m)[near])), 2)
print("Median walk / straight line, for hexagons with the service within 1.2 km in a straight line:")
display(pd.Series(detour).rename("times").to_frame().T)

Median walk / straight line, for hexagons with the service within 1.2 km in a straight line:


,healthcare,school,park,grocery,rail
times,1.71,1.9,1.88,1.71,1.69


## Step 6: save

In [9]:
result.to_parquet(PROCESSED / "hex_access.parquet", index=False)

summary = {
    "osm_date": meta["osm_date"][:10],
    "walk_speed_kmh": WALK_SPEED_KMH, "threshold_min": THRESHOLD_MIN,
    "people": int(people),
    "share_by_score": {int(k): round(float(v), 1) for k, v in by_score.items()},
    "share_within": {c: round(float(v), 1) for c, v in share.items()},
    "detour_median": detour,
}
(PROCESSED / "access_summary.json").write_text(json.dumps(summary, indent=1))
print("Saved hex_access.parquet and access_summary.json")
print(json.dumps(summary, indent=1))

Saved hex_access.parquet and access_summary.json
{
 "osm_date": "2026-10-06",
 "walk_speed_kmh": 4.8,
 "threshold_min": 15,
 "people": 8343652,
 "share_by_score": {
  "0": 15.5,
  "1": 14.6,
  "2": 15.7,
  "3": 19.0,
  "4": 25.1,
  "5": 10.1
 },
 "share_within": {
  "healthcare": 46.8,
  "school": 57.1,
  "park": 65.6,
  "grocery": 66.5,
  "rail": 17.9
 },
 "detour_median": {
  "healthcare": 1.71,
  "school": 1.9,
  "park": 1.88,
  "grocery": 1.71,
  "rail": 1.69
 }
}


___
**END**